In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir('/content/drive/MyDrive')

!mkdir -p olist-customer-intelligence/{data/raw,data/processed,notebooks,sql,src,models,reports,app/pages,docs}
os.chdir('/content/drive/MyDrive/M5-forecasting')
!touch src/__init__.py
print("Now in:", os.getcwd())

Now in: /content/drive/MyDrive/M5-forecasting


In [3]:
%%writefile src/features.py
"""Feature engineering for M5 forecasting. All temporal features are
computed per-series (groupby id) to prevent leakage across series boundaries."""
import pandas as pd
import numpy as np


def add_lags(df, lags=(1, 7, 28)):
    g = df.groupby("id", observed=True)["sales"]
    for l in lags:
        df[f"lag_{l}"] = g.shift(l).astype("float32")
    return df


def add_rolling(df, windows=(7, 14, 28, 56), base_shift=1):
    """Rolling stats on shifted sales (no same-day leakage)."""
    shifted = df.groupby("id", observed=True)["sales"].shift(base_shift)
    df["_shifted"] = shifted
    g = df.groupby("id", observed=True)["_shifted"]
    for w in windows:
        df[f"roll_mean_{w}"] = g.transform(lambda s: s.rolling(w, min_periods=1).mean()).astype("float32")
        df[f"roll_std_{w}"]  = g.transform(lambda s: s.rolling(w, min_periods=1).std()).astype("float32")
    # zero-demand density: fraction of zero days in trailing window (intermittency signal)
    for w in (28, 56):
        df[f"roll_zero_frac_{w}"] = g.transform(
            lambda s: s.eq(0).rolling(w, min_periods=1).mean()
        ).astype("float32")
    return df.drop(columns=["_shifted"])


def add_calendar(df):
    df["dow"] = df["date"].dt.dayofweek.astype("int8")
    df["dom"] = df["date"].dt.day.astype("int8")
    df["week"] = df["date"].dt.isocalendar().week.astype("int8")
    # cyclical encodings
    df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7).astype("float32")
    df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7).astype("float32")
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12).astype("float32")
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12).astype("float32")
    df["is_weekend"] = df["dow"].isin([5, 6]).astype("int8")
    return df


def add_events(df):
    df["is_event"] = df["event_name_1"].notna().astype("int8")
    # Christmas: the chain-wide closure days
    df["is_christmas"] = ((df["month"] == 12) & (df["dom"] == 25)).astype("int8")
    # snap already present as int8
    return df


def add_price_features(df):
    g = df.groupby("id", observed=True)["sell_price"]
    df["price_lag_1"] = g.shift(1).astype("float32")
    df["price_change"] = (df["sell_price"] / df["price_lag_1"] - 1).astype("float32")
    # price vs each item's own historical mean (promo signal)
    df["price_rel_mean"] = (df["sell_price"] / g.transform("mean")).astype("float32")
    df["price_momentum"] = g.transform(
        lambda s: s.shift(1).rolling(28, min_periods=1).mean()
    ).astype("float32")
    return df


def build_features(df):
    df = df.sort_values(["id", "date"]).reset_index(drop=True)
    df = add_lags(df)
    df = add_rolling(df)
    df = add_calendar(df)
    df = add_events(df)
    df = add_price_features(df)
    return df

Overwriting src/features.py


In [4]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np
from src.features import build_features

In [5]:
long = pd.read_parquet("data/processed/sales_long.parquet")
print(long.shape)

(22257700, 20)


In [6]:
feat = build_features(long)
print(feat.shape)
print(feat.memory_usage(deep=True).sum() / 1e9, "GB")

(22257700, 47)
6.741822014 GB


In [7]:
feat.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,dow_cos,month_sin,month_cos,is_weekend,is_event,is_christmas,price_lag_1,price_change,price_rel_mean,price_momentum
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1212,0,2014-05-24,11417,...,-0.222521,0.5,-0.866025,1,0,0,NaN,NaN,1.0,NaN
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1213,0,2014-05-25,11417,...,0.623490,0.5,-0.866025,1,0,0,2.24,-4.257474e-09,1.0,2.24
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1214,0,2014-05-26,11417,...,1.000000,0.5,-0.866025,0,1,0,2.24,-4.257474e-09,1.0,2.24
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1215,0,2014-05-27,11417,...,0.623490,0.5,-0.866025,0,0,0,2.24,-4.257474e-09,1.0,2.24
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1216,0,2014-05-28,11417,...,-0.222521,0.5,-0.866025,0,0,0,2.24,-4.257474e-09,1.0,2.24


In [8]:
first_rows = feat.groupby("id", observed=True).head(1)
print("series-start lag_1 non-null :", first_rows["lag_1"].notna().sum())

series-start lag_1 non-null : 0


In [9]:
import gc

# free the pre-feature frame if still around
try:
    del long
except NameError:
    pass
gc.collect()

# boolean-mask filter without reset_index (reset copies the whole frame)
feat = feat.loc[feat["lag_28"].notna()]
print("after warmup drop:", feat.shape)


# downcast any stragglers
f64 = feat.select_dtypes("float64").columns
feat[f64] = feat[f64].astype("float32")
i64 = feat.select_dtypes("int64").columns
feat[i64] = feat[i64].apply(pd.to_numeric, downcast="integer")
print(feat.memory_usage(deep=True).sum() / 1e9, "GB after downcast")

feat.to_parquet(
    "data/processed/features.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy",
    row_group_size=500_000,
)
print("saved")

after warmup drop: (21403980, 47)
6.440555162 GB after downcast
saved


## Feature Engineering

**Output:** `features.parquet` — 21.4M rows × 47 cols, 6.4 GB
(dropped first 28 days/series as lag warmup)

**Features built (all per-series via groupby id — no cross-boundary leakage):**
- **Lags:** 1, 7, 28
- **Rolling stats:** mean & std over 7/14/28/56, all computed on `sales.shift(1)` (no same-day leakage)
- **Intermittency:** trailing zero-fraction over 28/56 (density of zero-demand days)
- **Calendar:** dow, dom, week, cyclical sin/cos (dow & month), is_weekend
- **Events:** is_event, is_christmas (chain-wide closure), snap
- **Price:** lag, pct change, price vs item mean (promo signal), 28d momentum

**Leakage guarantees verified:**
- Series-start lag_1 non-null count = **0** (no bleed across item boundaries)
- Rolling features use shift(1) → never include current day

**Design notes:**
- shift-before-roll is the core anti-leakage rule
- 730-day window leaves ample history after 28-day warmup drop
